# Week 19 -- DPO and the Preference Zoo -- Workbook
Each exercise is a triple: prompt -- hint -- scaffold -- hidden test that uses the reference solution.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import torch, torch.nn.functional as F
torch.set_num_threads(1)
from project import solutions as S
from project.losses import dpo_loss, ipo_loss, simpo_loss, orpo_loss, _log1mexp, implicit_reward_margin
from project.data import ToyPreferenceDataset

## Exercise 1 (star) -- the implicit reward
Given `logp = log pi(y|x)` and `logp_ref = log pi_ref(y|x)`, return the implicit reward $\hat r = \beta(\log\pi - \log\pi_\text{ref})$.

> **Hint**: derive it from the closed-form RLHF optimum of week 18; the partition function disappears in any pairwise difference.

In [ ]:
def implicit_reward(logp, logp_ref, beta):
    raise NotImplementedError

In [ ]:
lp = torch.tensor([0.0, -2.0]); lref = torch.tensor([-1.0, -1.0])
r = S.exercise_1_implicit_reward(lp, lref, beta=0.5)
assert torch.allclose(r, torch.tensor([0.5, -0.5]))
print('implicit reward =', r)

## Exercise 2 (star star) -- DPO loss from scratch

> **Hint**: `-log sigmoid(beta * ((logp_w - logp_ref_w) - (logp_l - logp_ref_l)))`.  Use `F.logsigmoid`.

In [ ]:
def my_dpo(logp_w, logp_l, logp_ref_w, logp_ref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
z = torch.zeros(4)
loss = S.exercise_2_dpo(z, z, z, z, beta=0.1)
assert abs(float(loss) - 0.693) < 1e-2
print('DPO loss at equal logits =', float(loss))

## Exercise 3 (star star star) -- IPO loss from scratch
Squared loss with target margin $1/(2\beta)$.  Explain in a comment why this avoids DPO's saturation pathology.

> **Hint**: `((lr_w - lr_l - 1/(2 beta)) ** 2).mean()`.

In [ ]:
def my_ipo(logp_w, logp_l, logp_ref_w, logp_ref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
z = torch.zeros(4)
loss = S.exercise_3_ipo(z, z, z, z, beta=0.5)
assert abs(float(loss) - 1.0) < 1e-5
print('IPO loss at margin 0, target 1.0 =', float(loss))

## Exercise 4 (star star) -- SimPO loss

> **Hint**: average the log probability per token: `logp / max(1, len)`.

In [ ]:
def my_simpo(logp_w, logp_l, len_w, len_l, beta=2.0, gamma=1.0):
    raise NotImplementedError

In [ ]:
lp_w = torch.tensor([-1.0]); lp_l = torch.tensor([-5.0]); L_w = torch.tensor([1]); L_l = torch.tensor([5])
out = S.exercise_4_simpo(lp_w, lp_l, L_w, L_l, beta=2.0, gamma=0.0)
assert abs(float(out) - 0.693) < 1e-2
print('SimPO loss at equal avg-logprob =', float(out))

## Exercise 5 (star star) -- numerically stable log(1 - exp(x))

> **Hint**: use `expm1` for x > log(1/2), otherwise `log1p(-exp(x))`.

In [ ]:
def my_log1mexp(x):
    raise NotImplementedError

In [ ]:
x = torch.tensor([-0.1, -1.0, -5.0])
out = S.exercise_5_log1mexp(x)
naive = torch.log1p(-torch.exp(x))
assert torch.allclose(out, naive, atol=1e-5)
print('log1mexp =', out)

## Exercise 6 (star star) -- ORPO loss

> **Hint**: NLL on winners + (-beta * log sigmoid) of the log-odds-ratio gap.

In [ ]:
def my_orpo(logp_w, logp_l, nll_w, beta=0.1):
    raise NotImplementedError

In [ ]:
lp_w = torch.tensor([-1.0]); lp_l = torch.tensor([-3.0]); nll_w = torch.tensor([1.0])
out = S.exercise_6_orpo(lp_w, lp_l, nll_w, beta=0.1)
assert torch.isfinite(out)
print('ORPO loss =', float(out))

## Exercise 7 (star star) -- win-rate over two reward arrays

> **Hint**: fraction of items where `rewards_a > rewards_b`.

In [ ]:
def my_win_rate(a, b):
    raise NotImplementedError

In [ ]:
a = torch.tensor([1.0, 2.0, 0.0]); b = torch.tensor([0.0, 1.0, 1.0])
rho = S.exercise_7_win_rate(a, b)
assert abs(rho - 2/3) < 1e-6
print('win rate =', rho)

## Exercise 8 (star star) -- implicit-reward margin diagnostic

> **Hint**: a one-line wrapper around the formula in week 18 exercise 7.

In [ ]:
def my_margin(logp_w, logp_l, lref_w, lref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
m = S.exercise_8_margin(
    torch.tensor([-1.0]), torch.tensor([-3.0]),
    torch.tensor([-2.0]), torch.tensor([-2.0]), beta=0.1
)
assert float(m) > 0
print('margin =', float(m))

## Exercise 9 (star star star) -- *Open-ended:* mini bake-off and length-bias diagnostic
Run a *very small* bake-off and inspect the average token length of chosen vs. rejected sequences from the toy dataset.  Larger gap => more length inflation potential.

> **Hint**: feed `lens_chosen` and `lens_rejected` (the response lengths) into `S.exercise_9_length_bias`.

In [ ]:
from project.data import encode
ds = ToyPreferenceDataset(n=32, seed=0)
lens_chosen   = torch.tensor([len(c) for (_, c, _) in ds.pairs])
lens_rejected = torch.tensor([len(l) for (_, _, l) in ds.pairs])
gap = S.exercise_9_length_bias(lens_chosen, lens_rejected)
print('length gap (chosen - rejected) =', gap)
# Negative gap means the *rejected* set is longer -- exactly the dataset we want SimPO on.

## Exercise 10 (star) -- *Meaningful objective:* Plackett--Luce DPO
Derive a DPO loss for a *ranking* over $k$ items, not just pairs.

Mathematical derivation (no code needed; check yourself against the answer below):

Given a ranked tuple $y_{(1)} \succ y_{(2)} \succ \dots \succ y_{(k)}$ for prompt $x$, the Plackett--Luce probability of that ranking under reward $r$ is

$$ P(y_{(1)} \succ \dots \succ y_{(k)}) = \prod_{i=1}^{k} \frac{\exp(r(x, y_{(i)}))}{\sum_{j=i}^{k} \exp(r(x, y_{(j)}))}. $$

Substituting $r = \beta(\log\pi_\theta - \log\pi_\text{ref})$ and taking the negative log gives the loss

$$ \mathcal L_{\text{PL-DPO}} = -\sum_i \left[ \beta(\log\pi_\theta(y_{(i)}|x) - \log\pi_\text{ref}(y_{(i)}|x)) - \log \sum_{j \ge i} e^{\beta(\log\pi_\theta(y_{(j)}|x) - \log\pi_\text{ref}(y_{(j)}|x))} \right]. $$

When $k=2$ this collapses to the pairwise DPO loss; for $k>2$ it generalises naturally and is the loss form used by *ranked-DPO* and the multi-candidate ATS suggestion-ranking setting.

## Wrap-up
You have implemented (or read) every DPO-family loss in the modern literature.
On a real GPU, `project/trl_recipe.py` (sketched in `notes.tex`) extends this to Qwen-2.5-0.5B + LoRA on a slice of UltraFeedback.